# 🏥 XFarmaCare — Feature Engineering & Modelo de Churn
## Motor Autónomo de Retención y Rentabilidad

**Objetivo:** Construir las features a nivel cliente a partir de datos transaccionales,
interacciones del call center y adherencia terapéutica. Luego entrenar un modelo
que prediga la probabilidad de que un cliente abandone en los próximos 30-60 días.

**Arquitectura:** Este notebook lee los CSVs del modelo estrella, crea un dataset
analítico unificado (una fila por cliente) y exporta:
- El modelo entrenado (.pkl)
- El scaler (.pkl)
- Las predicciones de churn (.csv) conectadas por `customer_id`

---

In [ ]:
# === Librerías necesarias ===
# Manejo de datos
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, f1_score)
from sklearn.pipeline import Pipeline
import joblib
import os

# Rutas de los datos
DATA_DIR = "data/"
MODEL_DIR = "models/"
OUTPUT_DIR = "outputs/"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Librerías cargadas correctamente.")

## 1. Carga de Datos del Modelo Estrella
Leemos las tablas de dimensiones y hechos. Todo se conecta por `customer_id`.

In [ ]:
# === Cargar todas las tablas del modelo estrella ===
clientes = pd.read_csv(f"{DATA_DIR}dim_clientes.csv")
productos = pd.read_csv(f"{DATA_DIR}dim_productos.csv")
sucursales = pd.read_csv(f"{DATA_DIR}dim_sucursales.csv")
transacciones = pd.read_csv(f"{DATA_DIR}fact_transacciones.csv", parse_dates=["fecha_compra"])
callcenter = pd.read_csv(f"{DATA_DIR}fact_interacciones_callcenter.csv", parse_dates=["fecha_interaccion"])
adherencia = pd.read_csv(f"{DATA_DIR}fact_adherencia_terapeutica.csv")
campanas = pd.read_csv(f"{DATA_DIR}fact_campanas_retencion.csv")

# Vista rápida de cada tabla
for nombre, df in [("Clientes", clientes), ("Productos", productos),
                    ("Transacciones", transacciones), ("Call Center", callcenter),
                    ("Adherencia", adherencia), ("Campañas", campanas)]:
    print(f"{nombre:20s} → {df.shape[0]:>8,} filas × {df.shape[1]:>3} columnas")

## 2. Exploración Rápida (EDA)
Revisamos distribuciones clave antes de construir features.

In [ ]:
# === Distribución de clientes por zona y tipo ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Por zona geográfica
clientes['zona'].value_counts().plot(kind='barh', ax=axes[0], color='#2E86AB')
axes[0].set_title('Clientes por Zona Geográfica')
axes[0].set_xlabel('Cantidad')

# Crónicos vs No crónicos
clientes['es_cronico'].value_counts().plot(kind='pie', ax=axes[1],
    labels=['No Crónico', 'Crónico'], colors=['#A8DADC', '#E63946'],
    autopct='%1.1f%%', startangle=90)
axes[1].set_title('Pacientes Crónicos')
axes[1].set_ylabel('')

# Distribución de edad
axes[2].hist(clientes['edad'], bins=30, color='#457B9D', edgecolor='white')
axes[2].set_title('Distribución de Edad')
axes[2].set_xlabel('Edad')
axes[2].axvline(clientes['edad'].mean(), color='red', linestyle='--', label=f"Media: {clientes['edad'].mean():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}eda_distribucion_clientes.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Análisis de transacciones por canal y tendencia temporal ===
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Ventas por canal
transacciones.groupby('canal_compra')['total_venta_dop'].sum().sort_values().plot(
    kind='barh', ax=axes[0], color='#1D3557')
axes[0].set_title('Ingreso Total por Canal de Compra (DOP)')
axes[0].set_xlabel('Total Ventas (DOP)')

# Tendencia mensual
ventas_mensuales = transacciones.set_index('fecha_compra').resample('M')['total_venta_dop'].sum()
ventas_mensuales.plot(ax=axes[1], color='#E63946', linewidth=2)
axes[1].set_title('Tendencia de Ventas Mensuales')
axes[1].set_ylabel('Ventas (DOP)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}eda_ventas.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Sentimiento del Call Center ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de sentimientos
sent_counts = callcenter['sentimiento'].value_counts()
colors_sent = {'POSITIVO': '#2A9D8F', 'NEUTRO': '#E9C46A', 'NEGATIVO': '#E76F51'}
sent_counts.plot(kind='bar', ax=axes[0], color=[colors_sent[s] for s in sent_counts.index])
axes[0].set_title('Distribución de Sentimientos - Call Center')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=0)

# Motivos de contacto más frecuentes
callcenter['motivo'].value_counts().head(8).plot(kind='barh', ax=axes[1], color='#264653')
axes[1].set_title('Top 8 Motivos de Contacto')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}eda_callcenter.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering
Construimos un dataset analítico con una fila por cliente. Cada feature se calcula
a partir de los datos transaccionales, de call center y de adherencia.

### Grupos de features:
1. **Comportamiento de compra:** frecuencia, recencia, monto, tendencia
2. **Sentimiento (call center):** score promedio, cantidad de quejas, tasa de resolución
3. **Adherencia terapéutica:** solo para crónicos, tasa promedio, gaps
4. **Perfil demográfico:** edad, zona, seguro, programa de lealtad

In [ ]:
# === FECHA DE CORTE para definir churn ===
# Usamos los últimos 60 días del dataset como ventana de observación
FECHA_CORTE = transacciones['fecha_compra'].max() - pd.Timedelta(days=60)
FECHA_MAX = transacciones['fecha_compra'].max()

print(f"Fecha máxima en datos: {FECHA_MAX.date()}")
print(f"Fecha de corte (60 días antes): {FECHA_CORTE.date()}")
print(f"Ventana de observación para churn: {FECHA_CORTE.date()} → {FECHA_MAX.date()}")

In [ ]:
# === 3.1 Features de Comportamiento de Compra ===

# Transacciones ANTES de la ventana de churn (datos de entrenamiento)
txn_historicas = transacciones[transacciones['fecha_compra'] < FECHA_CORTE].copy()

# Agrupar por cliente
feat_compras = txn_historicas.groupby('customer_id').agg(
    total_transacciones=('transaction_id', 'count'),
    total_gastado_dop=('total_venta_dop', 'sum'),
    promedio_ticket_dop=('total_venta_dop', 'mean'),
    max_ticket_dop=('total_venta_dop', 'max'),
    productos_distintos=('product_id', 'nunique'),
    canales_usados=('canal_compra', 'nunique'),
    primera_compra=('fecha_compra', 'min'),
    ultima_compra=('fecha_compra', 'max'),
    descuento_promedio=('descuento_porcentaje', 'mean'),
    compras_receta_recurrente=('fue_receta_recurrente', 'sum')
).reset_index()

# Recencia: días desde la última compra hasta la fecha de corte
feat_compras['recencia_dias'] = (FECHA_CORTE - feat_compras['ultima_compra']).dt.days

# Antigüedad: días desde la primera compra
feat_compras['antiguedad_dias'] = (FECHA_CORTE - feat_compras['primera_compra']).dt.days

# Frecuencia: transacciones por mes activo
feat_compras['frecuencia_mensual'] = (
    feat_compras['total_transacciones'] / np.maximum(feat_compras['antiguedad_dias'] / 30, 1)
).round(2)

# Canal dominante
canal_dom = txn_historicas.groupby('customer_id')['canal_compra'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
canal_dom.columns = ['customer_id', 'canal_dominante']
feat_compras = feat_compras.merge(canal_dom, on='customer_id', how='left')

# Tendencia: comparar gasto de los últimos 3 meses vs los 3 anteriores
fecha_3m = FECHA_CORTE - pd.Timedelta(days=90)
fecha_6m = FECHA_CORTE - pd.Timedelta(days=180)

gasto_reciente = txn_historicas[txn_historicas['fecha_compra'] >= fecha_3m].groupby('customer_id')['total_venta_dop'].sum()
gasto_anterior = txn_historicas[(txn_historicas['fecha_compra'] >= fecha_6m) & (txn_historicas['fecha_compra'] < fecha_3m)].groupby('customer_id')['total_venta_dop'].sum()

tendencia = pd.DataFrame({'gasto_reciente': gasto_reciente, 'gasto_anterior': gasto_anterior}).fillna(0)
tendencia['tendencia_gasto'] = np.where(
    tendencia['gasto_anterior'] > 0,
    ((tendencia['gasto_reciente'] - tendencia['gasto_anterior']) / tendencia['gasto_anterior']).clip(-1, 5),
    0
)
feat_compras = feat_compras.merge(tendencia[['tendencia_gasto']], left_on='customer_id', right_index=True, how='left')
feat_compras['tendencia_gasto'] = feat_compras['tendencia_gasto'].fillna(0)

# Limpiar columnas de fecha que no necesitamos como features
feat_compras = feat_compras.drop(columns=['primera_compra', 'ultima_compra'])

print(f"Features de compra: {feat_compras.shape}")
feat_compras.head(3)

In [ ]:
# === 3.2 Features de Sentimiento (Call Center) ===

# Interacciones antes de la fecha de corte
cc_hist = callcenter[callcenter['fecha_interaccion'] < FECHA_CORTE].copy()

feat_sentiment = cc_hist.groupby('customer_id').agg(
    total_interacciones=('interaction_id', 'count'),
    interacciones_negativas=('sentimiento', lambda x: (x == 'NEGATIVO').sum()),
    interacciones_positivas=('sentimiento', lambda x: (x == 'POSITIVO').sum()),
    puntaje_neg_promedio=('puntaje_negativo', 'mean'),
    puntaje_pos_promedio=('puntaje_positivo', 'mean'),
    estrellas_promedio=('estrellas', 'mean'),
    tasa_resolucion=('resuelto', 'mean'),
    tiempo_espera_promedio=('tiempo_espera_seg', 'mean'),
    duracion_promedio_min=('duracion_minutos', 'mean')
).reset_index()

# Proporción de interacciones negativas sobre el total
feat_sentiment['ratio_negatividad'] = (
    feat_sentiment['interacciones_negativas'] / feat_sentiment['total_interacciones']
).round(3)

# Motivo más frecuente por cliente
motivo_freq = cc_hist.groupby('customer_id')['motivo'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
motivo_freq.columns = ['customer_id', 'motivo_mas_frecuente']
feat_sentiment = feat_sentiment.merge(motivo_freq, on='customer_id', how='left')

print(f"Features de sentimiento: {feat_sentiment.shape}")
feat_sentiment.head(3)

In [ ]:
# === 3.3 Features de Adherencia Terapéutica ===
# Solo aplica a pacientes crónicos

feat_adherencia = adherencia.groupby('customer_id').agg(
    tasa_adherencia_promedio=('tasa_adherencia_periodo', 'mean'),
    tasa_adherencia_min=('tasa_adherencia_periodo', 'min'),
    dias_gap_promedio=('dias_gap', 'mean'),
    dias_gap_max=('dias_gap', 'max'),
    total_ciclos_tratamiento=('adherencia_id', 'count'),
    ciclos_con_retraso=('dias_gap', lambda x: (x > 7).sum())
).reset_index()

# Proporción de ciclos con retraso significativo
feat_adherencia['ratio_retraso'] = (
    feat_adherencia['ciclos_con_retraso'] / feat_adherencia['total_ciclos_tratamiento']
).round(3)

# Tendencia de adherencia: comparar últimos 3 ciclos vs anteriores
def adherencia_trend(group):
    if len(group) < 4:
        return 0
    recientes = group['tasa_adherencia_periodo'].tail(3).mean()
    anteriores = group['tasa_adherencia_periodo'].iloc[:-3].mean()
    return round(recientes - anteriores, 3)

trend_adh = adherencia.groupby('customer_id').apply(adherencia_trend).reset_index()
trend_adh.columns = ['customer_id', 'tendencia_adherencia']
feat_adherencia = feat_adherencia.merge(trend_adh, on='customer_id', how='left')

print(f"Features de adherencia: {feat_adherencia.shape}")
feat_adherencia.head(3)

In [ ]:
# === 3.4 Unir todo en un dataset analítico ===

# Base: todos los clientes
df_modelo = clientes[['customer_id', 'edad', 'sexo', 'provincia', 'zona',
                       'tipo_seguro', 'es_cronico', 'condicion_cronica',
                       'programa_lealtad', 'nivel_lealtad', 'canal_preferido',
                       'ingreso_estimado']].copy()

# Merge con features
df_modelo = df_modelo.merge(feat_compras, on='customer_id', how='left')
df_modelo = df_modelo.merge(feat_sentiment, on='customer_id', how='left')
df_modelo = df_modelo.merge(feat_adherencia, on='customer_id', how='left')

# Rellenar NaN para clientes sin interacciones o sin adherencia
cols_sentiment = ['total_interacciones', 'interacciones_negativas', 'interacciones_positivas',
                  'puntaje_neg_promedio', 'puntaje_pos_promedio', 'estrellas_promedio',
                  'tasa_resolucion', 'tiempo_espera_promedio', 'duracion_promedio_min',
                  'ratio_negatividad']
for col in cols_sentiment:
    df_modelo[col] = df_modelo[col].fillna(0)

cols_adherencia = ['tasa_adherencia_promedio', 'tasa_adherencia_min', 'dias_gap_promedio',
                   'dias_gap_max', 'total_ciclos_tratamiento', 'ciclos_con_retraso',
                   'ratio_retraso', 'tendencia_adherencia']
for col in cols_adherencia:
    df_modelo[col] = df_modelo[col].fillna(0)

# Motivo más frecuente
df_modelo['motivo_mas_frecuente'] = df_modelo['motivo_mas_frecuente'].fillna('Sin contacto')

# Rellenar compras (clientes sin transacciones = ya se fueron antes)
df_modelo['total_transacciones'] = df_modelo['total_transacciones'].fillna(0)
df_modelo['total_gastado_dop'] = df_modelo['total_gastado_dop'].fillna(0)
df_modelo['recencia_dias'] = df_modelo['recencia_dias'].fillna(999)

print(f"Dataset analítico final: {df_modelo.shape}")
print(f"\nColumnas: {list(df_modelo.columns)}")
df_modelo.head()

## 4. Definición de la Variable Objetivo (Churn)
Un cliente se considera "churner" si **no realizó ninguna compra** en los últimos
60 días del dataset. Para crónicos, usamos un criterio más estricto: si dejó de
comprar su medicamento habitual por más de 45 días.

In [ ]:
# === Crear etiqueta de churn ===

# Compras en la ventana de observación (últimos 60 días)
txn_ventana = transacciones[transacciones['fecha_compra'] >= FECHA_CORTE]
clientes_activos_ventana = set(txn_ventana['customer_id'].unique())

# Churn base: no compró en los últimos 60 días
df_modelo['churn'] = (~df_modelo['customer_id'].isin(clientes_activos_ventana)).astype(int)

# Para crónicos: si compró pero no su medicamento habitual, también es señal
# (esto ya está capturado implícitamente por la recencia y adherencia)

# Ajustar: clientes sin historial de compras (muy nuevos) no son churn
df_modelo.loc[df_modelo['total_transacciones'] == 0, 'churn'] = 0

# Resumen
print("Distribución de Churn:")
print(df_modelo['churn'].value_counts())
print(f"\nTasa de churn: {df_modelo['churn'].mean()*100:.1f}%")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_modelo['churn'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#2A9D8F', '#E76F51'])
axes[0].set_title('Distribución de Churn')
axes[0].set_xticklabels(['Activo', 'Churner'], rotation=0)
axes[0].set_ylabel('Cantidad')

# Churn por segmento crónico
churn_by_cronico = df_modelo.groupby('es_cronico')['churn'].mean() * 100
churn_by_cronico.plot(kind='bar', ax=axes[1], color=['#457B9D', '#E63946'])
axes[1].set_title('Tasa de Churn: Crónicos vs No Crónicos')
axes[1].set_xticklabels(['No Crónico', 'Crónico'], rotation=0)
axes[1].set_ylabel('% Churn')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}distribucion_churn.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Preparación del Modelo
Codificamos variables categóricas y separamos train/test.

In [ ]:
# === Codificar variables categóricas ===
le_dict = {}  # Guardar los encoders para el scoring diario

cols_categoricas = ['sexo', 'zona', 'tipo_seguro', 'condicion_cronica',
                    'nivel_lealtad', 'canal_preferido', 'ingreso_estimado',
                    'canal_dominante', 'motivo_mas_frecuente']

for col in cols_categoricas:
    if col in df_modelo.columns:
        df_modelo[col] = df_modelo[col].fillna('Desconocido')
        le = LabelEncoder()
        df_modelo[f'{col}_enc'] = le.fit_transform(df_modelo[col])
        le_dict[col] = le

# Columnas que NO entran al modelo
cols_excluir = ['customer_id', 'provincia', 'es_cronico', 'programa_lealtad'] + cols_categoricas

# Features numéricas + las codificadas
feature_cols = [c for c in df_modelo.columns if c not in cols_excluir + ['churn']]

# Convertir booleanos a int
for col in feature_cols:
    if df_modelo[col].dtype == 'bool':
        df_modelo[col] = df_modelo[col].astype(int)

X = df_modelo[feature_cols].copy()
y = df_modelo['churn'].copy()

# Rellenar cualquier NaN restante
X = X.fillna(0)

print(f"Features para el modelo: {X.shape[1]}")
print(f"Distribución target: {y.value_counts().to_dict()}")
print(f"\nFeatures seleccionadas:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# === Split train/test estratificado ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Escalar features numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Churn en Train: {y_train.mean()*100:.1f}% | Churn en Test: {y_test.mean()*100:.1f}%")

## 6. Entrenamiento del Modelo
Probamos 3 algoritmos y nos quedamos con el mejor por AUC-ROC.

In [ ]:
# === Entrenar y comparar modelos ===
modelos = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=20,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        min_samples_leaf=20, random_state=42
    )
}

resultados = {}

for nombre, modelo in modelos.items():
    print(f"\nEntrenando: {nombre}...")
    
    # Usar datos escalados para Logistic, originales para trees
    if 'Logistic' in nombre:
        modelo.fit(X_train_scaled, y_train)
        y_pred = modelo.predict(X_test_scaled)
        y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    else:
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    
    resultados[nombre] = {
        'modelo': modelo,
        'auc_roc': auc,
        'f1_score': f1,
        'y_proba': y_proba
    }
    
    print(f"  AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f}")

# Seleccionar el mejor modelo
mejor_nombre = max(resultados, key=lambda x: resultados[x]['auc_roc'])
mejor = resultados[mejor_nombre]
print(f"\n{'='*50}")
print(f"MEJOR MODELO: {mejor_nombre}")
print(f"AUC-ROC: {mejor['auc_roc']:.4f} | F1: {mejor['f1_score']:.4f}")
print(f"{'='*50}")

In [ ]:
# === Curva ROC de los 3 modelos ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for nombre, res in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{nombre} (AUC={res['auc_roc']:.3f})", linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC — Comparación de Modelos')
axes[0].legend()

# Matriz de confusión del mejor modelo
if 'Logistic' in mejor_nombre:
    y_pred_final = mejor['modelo'].predict(X_test_scaled)
else:
    y_pred_final = mejor['modelo'].predict(X_test)

cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Activo', 'Churner'], yticklabels=['Activo', 'Churner'])
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Realidad')
axes[1].set_title(f'Matriz de Confusión — {mejor_nombre}')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}modelo_churn_evaluacion.png", dpi=150, bbox_inches='tight')
plt.show()

# Reporte detallado
print(f"\nReporte de Clasificación — {mejor_nombre}:")
print(classification_report(y_test, y_pred_final, target_names=['Activo', 'Churner']))

In [ ]:
# === Feature Importance ===
if hasattr(mejor['modelo'], 'feature_importances_'):
    importancias = pd.DataFrame({
        'feature': feature_cols,
        'importancia': mejor['modelo'].feature_importances_
    }).sort_values('importancia', ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    importancias.head(20).plot(kind='barh', x='feature', y='importancia',
                               ax=ax, color='#1D3557', legend=False)
    ax.set_title(f'Top 20 Features Más Importantes — {mejor_nombre}')
    ax.set_xlabel('Importancia')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}feature_importance_churn.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nTop 10 features:")
    for _, row in importancias.head(10).iterrows():
        print(f"  {row['feature']:35s} → {row['importancia']:.4f}")

## 7. Exportar Modelo y Predicciones
Guardamos el modelo entrenado, el scaler y las predicciones en archivos independientes.

In [ ]:
# === Guardar modelo, scaler y encoders ===
joblib.dump(mejor['modelo'], f"{MODEL_DIR}modelo_churn_xfarmacare.pkl")
joblib.dump(scaler, f"{MODEL_DIR}scaler_churn.pkl")
joblib.dump(le_dict, f"{MODEL_DIR}label_encoders_churn.pkl")
joblib.dump(feature_cols, f"{MODEL_DIR}feature_cols_churn.pkl")

print("Modelo guardado: modelo_churn_xfarmacare.pkl")
print("Scaler guardado: scaler_churn.pkl")
print("Encoders guardados: label_encoders_churn.pkl")
print("Feature cols guardadas: feature_cols_churn.pkl")

In [ ]:
# === Generar predicciones para TODOS los clientes ===
X_todos = df_modelo[feature_cols].fillna(0)

if 'Logistic' in mejor_nombre:
    X_todos_proc = scaler.transform(X_todos)
else:
    X_todos_proc = X_todos

proba_churn = mejor['modelo'].predict_proba(X_todos_proc)[:, 1]

# Crear CSV de salida conectado por customer_id
output_churn = pd.DataFrame({
    'customer_id': df_modelo['customer_id'],
    'probabilidad_churn': proba_churn.round(4),
    'riesgo_churn': pd.cut(proba_churn,
        bins=[0, 0.20, 0.40, 0.60, 0.80, 1.0],
        labels=['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Crítico']),
    'churn_real': df_modelo['churn'],
    'fecha_scoring': pd.Timestamp.now().strftime('%Y-%m-%d')
})

output_churn.to_csv(f"{OUTPUT_DIR}output_churn_scores.csv", index=False)
print(f"\nPredicciones exportadas: {len(output_churn):,} clientes")
print(f"Distribución de riesgo:")
print(output_churn['riesgo_churn'].value_counts().sort_index())

# También guardar el dataset analítico completo para R
df_modelo.to_csv(f"{OUTPUT_DIR}dataset_analitico_clientes.csv", index=False)
print(f"\nDataset analítico guardado: {df_modelo.shape}")

## ✅ Notebook Completado
- **Modelo entrenado** y exportado a `models/modelo_churn_xfarmacare.pkl`
- **Predicciones** exportadas a `outputs/output_churn_scores.csv`
- **Dataset analítico** en `outputs/dataset_analitico_clientes.csv` (para R)
- Todo conectado por `customer_id` para el modelo estrella

**Siguiente paso:** `02_modelo_churn_scoring_diario.ipynb` para el scoring recurrente.